In [ ]:
# Colab only: clone (or update) the repo on the joe branch and move into the project root
import os
import sys

if 'google.colab' in sys.modules:
    REPO = '/content/convolutional_vae_quishing'
    BRANCH = 'joe'
    if not os.path.exists(REPO):
        !git clone -b {BRANCH} https://github.com/AnhJoe/convolutional_vae_quishing.git {REPO}
    else:
        !git -C {REPO} fetch origin
        !git -C {REPO} checkout {BRANCH}
        !git -C {REPO} pull origin {BRANCH}
    os.chdir(REPO)

In [ ]:
#| output: false
# Local only: find the project root (the folder containing src/) and add it to the import path
# For local runs. Else if running Colab, skip this.

import sys
from pathlib import Path

# Set to root for src load
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

if not (root / "src").exists():
    raise FileNotFoundError("Could not find 'src' folder above current working directory.")

sys.path.insert(0, str(root))
print("Project root:", root)

# Supervised reference

This model is **context, not a contribution**: the course asks for a generative model, and the VAE is that model. The reference answers one question: how much do labels and malicious examples add over a model that only saw benign URLs?

It is the VAE's encoder (the same network, with the same sizes) plus one linear layer that outputs the chance that a URL is malicious. Reusing the encoder keeps the capacity the same, so a gap to the VAE does not come from a bigger network. It is trained from scratch on the benign **and** malicious train URLs with binary cross-entropy (the usual loss for a yes or no label), and the epoch with the lowest validation loss is kept. Its score is `prob_malicious`.

**Read its results with three caveats.**

1. **The training data differs.** The VAE and the floors train on benign train URLs only. The reference trains on benign and malicious train URLs (more than twice as many). The gap between the VAE and its reference therefore measures what labels and malicious examples add, and part of it can come from simply having more data. Word it that way, not as "supervision alone".
2. **The reference can use the shortcuts directly.** A labeled model can learn that a missing scheme points to benign and that an IP host points to malicious (the scheme alone has an AUROC of 0.77, and IP hosts are 99.5% to 100% malicious, see `1_eda.ipynb` section 4). A high reference score may therefore mostly reflect those shortcuts. Read it next to floor B and within the scheme strata, and do not describe it as an upper bound on what the URL carries.
3. **Its model selection uses labels.** The epoch is chosen on validation cross-entropy, which uses the validation labels. The VAE's epoch is chosen without any label.

The code lives in `src/supervised.py` and `src/runner.py`. Train the VAE first (`4_url_vae.ipynb`) so the reference can copy its encoder sizes.

In [ ]:
# Imports and settings
import json

from src import checks
from src.plots import plot_classifier_history
from src.prepare_data import load_items
from src.runner import matching_reference_config, run_supervised, trial_setup
from src.splits import SPLIT_NAMES

PROC_DIR = root / "data" / "processed"
RESULTS_DIR = root / "results"
ARTIFACTS_DIR = root / "data" / "artifacts"     # model weights and training histories (git-ignored)

## 1. Data

`load_items` reads the items table built by `2_data_processing.ipynb` (one row per kept URL, with its label, split and URL features). The data checks below confirm that the table is the one everyone on the team should have.

In [ ]:
# Load the items table and its summary, then count the URLs per split and class
items = load_items(PROC_DIR)
summary = json.loads((PROC_DIR / "items_summary.json").read_text())
items.groupby(["split", "cls"]).size().unstack().loc[SPLIT_NAMES]

**Checks on the data.** The next cell runs a set of checks on the items table and prints one row per check. Each row is `pass` or `FAIL`, and the cell stops with an error if any check fails, so a broken table cannot reach a model by accident. The checks confirm that the table matches `items_summary.json` (the row counts and the data version, which everyone on the team should see as the same value), that every split has both classes, that no registered domain sits in two splits of the same class, and that the URL columns are consistent.

In [ ]:
# Run the data checks on the items table (stops with an error if any check fails)
checks.data_checks(items, summary).report()

## 2. Settings

The encoder sizes are read from the trained VAE, so the two always match. The training settings (epochs, learning rate) can be changed here. We run one seed (seed 0). `QUICK = True` runs a two-epoch trial on a small slice and saves it under `results/quick`.

In [ ]:
# Settings: the reference copies the VAE encoder sizes. QUICK runs a short trial on a small slice
QUICK = False
SEEDS = [0]                      # one seed
run_items, res_dir, art_dir = trial_setup(items, RESULTS_DIR, ARTIFACTS_DIR, QUICK)
cfg = matching_reference_config(art_dir, epochs=2 if QUICK else 6)
cfg

## 3. Train

One classifier is trained for seed 0 (the one seed we run, set in `SEEDS`) and its scores are saved to `results/url_supervised/prob_malicious/seed<k>/`. A seed is skipped only when its scores exist and its run record (`run.json`: settings and data version) matches the current ones (the record also holds the environment and the checksums of the source CSVs, which are not compared). A changed setting or different data retrains it, and the log says what changed (`overwrite=True` retrains everything).

In [ ]:
# Train the supervised reference and save its scores (prints one line per epoch)
metrics, histories = run_supervised(run_items, res_dir, art_dir, cfg, SEEDS, sources=summary.get("sources"))

## 4. Training curves

Solid lines are the validation cross-entropy and dashed lines the training cross-entropy. A validation line that turns upward while the training line keeps falling means the model is starting to memorize the training URLs. The lowest validation point marks the epoch that is kept.

In [ ]:
# Plot the training and validation cross-entropy per epoch, one line per seed
plot_classifier_history(histories);

## 5. Test metrics

One row (seed 0). The comparison with the VAE and the floors is in `6_results.ipynb`.

In [ ]:
# Test metrics of every seed (see 3_floors.ipynb for what each column means)
metrics